# 12 — Inequality indices SII / RII (Stage 7c)

The headline RII for buffer- and route-based exposure, plus the **headline ratio** `RII_route / RII_buffer` with bootstrap CI. This is the H2 number that goes in the abstract.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from schools_sunbeds import audit, config, stats

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")

In [ ]:
panel = pd.read_parquet(sorted(config.DATA_PROCESSED.glob("exposure_panel_*.parquet"))[-1])
df = panel.dropna(subset=["imd_quintile"]).copy()
df["imd_quintile"] = df["imd_quintile"].astype("int64")
df["ridit_imd"] = stats.compute_ridit(df["imd_quintile"], df.get("sum_pupil_50m"))
df["_pupil_offset"] = df["sum_pupil_50m"].clip(lower=1)
df_h1 = df.loc[df["sum_pupil_50m"] > 0].copy()
print(f"All schools: {len(df)} | with-routes: {len(df_h1)}")

## 1. SII (population-weighted) for buffer + route

In [ ]:
sii_buffer = stats.slope_index_inequality(
    df["n_buffer_800m"], df["ridit_imd"], df["n_pupils_school"].fillna(1)
)
sii_route = stats.slope_index_inequality(
    df_h1["mean_per_pupil_route_50m"], df_h1["ridit_imd"], df_h1["sum_pupil_50m"]
)
print(f"SII buffer 800 m  (count diff Q1 vs Q5): {sii_buffer:+.3f}")
print(f"SII route  50 m   (per-pupil diff Q1 vs Q5): {sii_route:+.4f}")
print("  (Negative SII => more exposure in most-deprived areas, since ridit=0 is Q1.)")

## 2. RII (NB GLM, ratio of fitted values at ridit=0 vs ridit=1)

In [ ]:
def rii_buffer(d: pd.DataFrame) -> float:
    return 1.0 / stats.relative_index_inequality(d, "n_buffer_800m", "ridit_imd")

def rii_route(d: pd.DataFrame) -> float:
    sub = d.loc[d["sum_pupil_50m"] > 0]
    return 1.0 / stats.relative_index_inequality(
        sub, "sum_route_50m", "ridit_imd", offset_col="_pupil_offset"
    )

# Note: relative_index_inequality returns predicted(ridit=0) / predicted(ridit=1)
# which is exp(beta * (0-1)) = exp(-beta). With ridit=0 = most deprived, RII<1
# means MORE exposure in most-deprived areas. Inverting (1/RII) gives the
# more conventional Q1/Q5 ratio above 1 for pro-deprived inequality.
rii_b_point = rii_buffer(df)
rii_r_point = rii_route(df)
print(f"RII buffer 800 m: {rii_b_point:.3f}")
print(f"RII route 50 m  : {rii_r_point:.3f}")
print(f"  (>1 means more exposure for most-deprived schools)")

## 3. Headline RII ratio with bootstrap CI

In [ ]:
def rii_ratio(d: pd.DataFrame) -> float:
    return rii_route(d) / rii_buffer(d)

boot = stats.bootstrap_statistic(
    df,
    rii_ratio,
    n_boot=config.BOOTSTRAP_N,
    seed=config.BOOTSTRAP_SEED,
    cluster_col="la_code_dfe",
)
print(f"Headline RII_route / RII_buffer:")
print(f"  Point estimate: {boot.point:.3f}")
print(f"  95% CI         : ({boot.ci_low:.3f}, {boot.ci_high:.3f})")
print(f"  Bootstrap iters: {boot.n_boot}/{config.BOOTSTRAP_N}")
supported = boot.point > 1 and boot.ci_low > 1
print(f"\n  H2 {'SUPPORTED' if supported else 'NOT supported'}")
print("  (>1 with CI excluding 1 => walking-route exposure shows a steeper deprivation gradient)")

## 4. Persist headline tables

In [ ]:
headline = pd.DataFrame(
    {
        "metric": ["SII_buffer_800m", "SII_route_50m", "RII_buffer_800m", "RII_route_50m", "RII_ratio (route/buffer)"],
        "point": [sii_buffer, sii_route, rii_b_point, rii_r_point, boot.point],
        "ci_low": [None, None, None, None, boot.ci_low],
        "ci_high": [None, None, None, None, boot.ci_high],
    }
)
out = config.OUTPUTS_TABLES / f"T4_inequality_indices_{TODAY}.csv"
headline.to_csv(out, index=False)
headline